This notebook largely follows the approach of ["An NLP-based approach to assessing a company's maturity level in the digital era" (2024) by Romano, Sperlì, and Vignali](https://www.sciencedirect.com/science/article/pii/S0957417424011588)

In [1]:
from collections import Counter
import re
import string

import nltk
nltk.download()
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer

import os
import sys

# Comment out all but the appropriate option
where_running = "Google Colab"
# where_running = "Local Machine"

if where_running == "Local Machine":
  module_path = os.path.abspath(os.path.join('..'))
  if not module_path in sys.path:
      sys.path.insert(0, module_path)
elif where_running == "Google Colab":
  dirpath = '/content/digi-inno-road-prod'
  if not os.path.isdir(dirpath):
    # TODO git pull
    !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git
  sys.path.insert(0,dirpath)

from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps
from innoprod.wrangling.wrangling_tools import is_non_empty

NLTK Downloader
---------------------------------------------------------------------------
    d) Download   l) List    u) Update   c) Config   h) Help   q) Quit
---------------------------------------------------------------------------
Downloader> d

Download which package (l=list; x=cancel)?
  Identifier> all


       | 
       | Downloading package abc to /root/nltk_data...
       |   Unzipping corpora/abc.zip.
       | Downloading package alpino to /root/nltk_data...
       |   Unzipping corpora/alpino.zip.
       | Downloading package averaged_perceptron_tagger to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger.zip.
       | Downloading package averaged_perceptron_tagger_eng to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
       | Downloading package averaged_perceptron_tagger_ru to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger_ru.zip.
       | Downloading package averaged_perceptron_tagger_rus to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger_rus.zip.
       | Downloading package basque_grammars to /root/nltk_data...
       |   Unzipping grammars/basque_grammars.zip.
       | Downloading package bcp47 to /root/nltk_d


---------------------------------------------------------------------------
    d) Download   l) List    u) Update   c) Config   h) Help   q) Quit
---------------------------------------------------------------------------
Downloader> q
Cloning into 'digi-inno-road-prod'...
remote: Enumerating objects: 296, done.
remote: Counting objects: 100% (296/296), done.
remote: Compressing objects: 100% (196/196), done.
remote: Total 296 (delta 160), reused 203 (delta 74), pack-reused 0 (from 0)
Receiving objects: 100% (296/296), 499.52 KiB | 3.04 MiB/s, done.
Resolving deltas: 100% (160/160), done.


In [2]:
data = get_sheet_dfs()
roadmaps_df = wrangle_roadmaps(data['Roadmaps'])

In [ ]:
cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis',
    'Key potential industry 4.0 solutions to address the identified problems/gaps',
    'Recommended Action Plan to utilise Industry 4.0 Technology',
    'Overview of qualitative benefits of recommended Action Plan (positive/negative)',
    'Skills and other requirements that will be needed to successfully implement the recommended Action Plan',
    'What has been your overall opinion of the support you have received in this programme? (Add comments)'
]

In [ ]:
responses_df = roadmaps_df[['Client ID'] + cols].melt(id_vars=['Client ID'], value_vars=cols, var_name='Question', value_name='Response')
responses_df = responses_df[is_non_empty(responses_df['Response'])]

responses_df

,Client ID,Question,Response
0,f3fff05d-1a28-e8f3-c5f6-670d1d0797e3,Summary review of Edge Digital diagnostic repo...,[REDACTED] business has completed an edge digi...
1,458079bc-a5ab-2055-d514-6733331e9c5f,Summary review of Edge Digital diagnostic repo...,[REDACTED] Score: 7 STATUS: Based on your resp...
2,0369B4F9-9E83-E83D-9E6B-BF82E264A2BA,Summary review of Edge Digital diagnostic repo...,The EDD has identified that the Company has in...
3,e9b5b5a2-1ba0-1d3a-a374-67ed061c1e40,Summary review of Edge Digital diagnostic repo...,Summary review of [REDACTED] diagnostic report...
4,052CB881-3557-DCFA-E472-0E55A5D04590,Summary review of Edge Digital diagnostic repo...,The business has rated itself at level 4 in te...
...,...,...,...
2414,3044E4BE-D41D-1AD3-7DFE-D22F7E559972,What has been your overall opinion of the supp...,No negatives to report. Very good process and ...
2415,0EC71AFB-5867-DE5F-5061-32AEAE4B24BF,What has been your overall opinion of the supp...,Good. The only thing to say is it is difficult...
2416,2346FC83-5B42-B90D-3BBB-16BFD156902E,What has been your overall opinion of the supp...,"as usual a simple process, managed for us by o..."
2417,2AA6320B-75FF-BABC-1E79-EB96B0AE650E,What has been your overall opinion of the supp...,excellent support as always. I had received su...


# Preprocessing

In [ ]:
acronyms_abbreviations = {
    'I4.0': 'industry 4.0',
    'tech': 'technology',
}

# This is an important step: phrases here will be treated as single tokens, so we can ensure that important concepts are not split up during tokenization.
# This is especially important for multi-word technical terms, which would otherwise be split into their individual words and lose their meaning.
phrases = [
    "business returns",
    "digital roadmap",
    "digital strategy",
    "industry 4.0",
    "strategic skills",
    "technical skills",
]

In [ ]:
stops = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

regexpr = "|".join(phrases) + "|[\\w']+"
tokenizer = RegexpTokenizer(regexpr)

def preprocess_text(text):
    cleaned_text = text.replace("[REDACTED]", "")
    for short_form, full_form in acronyms_abbreviations.items():
        matches = re.findall(f'({short_form})[\\s|{re.escape(string.punctuation)}]', cleaned_text)
        for match in matches:
            cleaned_text = re.sub(re.escape(match), full_form, cleaned_text, flags=re.IGNORECASE)
    # Skipped removing punctuation as tokenization already does this.
    cleaned_text = cleaned_text.lower()
    # TO DO: explore switching the order of the lemmatization and stop word removal steps:
    # e.g., "has" is a stop word, but post-lemmatization it becomes "ha", which is not a stop word, so it is retained in the final output.
    tokenized_answer = tokenizer.tokenize(cleaned_text)
    lemmatized_answer = [lemmatizer.lemmatize(word) for word in tokenized_answer]
    final_answer = [lemma for lemma in lemmatized_answer if lemma not in stops]
    return final_answer

In [ ]:
'has' in stops

True

In [ ]:
responses_df['Cleaned Response'] = responses_df.apply(lambda row: preprocess_text(row['Response']), axis=1)

In [ ]:
all_words = responses_df['Cleaned Response'].explode()

# Expert supervision

In [ ]:
concepts = {}

def add_word(word, concept):
    if concept not in concepts:
        concepts[concept] = set()
    concepts[concept].add(word)

In [ ]:
counted_words = Counter(all_words).most_common()

In [ ]:
counted_words[0:20]


[('business', 3253),
 ('system', 2734),
 ('digital', 2621),
 ('data', 2344),
 ('management', 2000),
 ('process', 1743),
 ('support', 1700),
 ('technology', 1570),
 ('company', 1433),
 ('ha', 1281),
 ('development', 1223),
 ('investment', 1109),
 ('production', 1015),
 ('also', 983),
 ('increased', 976),
 ('implementation', 966),
 ('skill', 944),
 ('need', 934),
 ('new', 891),
 ('level', 844)]

In [ ]:
[(word, freq) for word, freq in counted_words if word in phrases]

[('digital strategy', 499),
 ('industry 4.0', 300),
 ('technical skills', 68),
 ('strategic skills', 57),
 ('business returns', 51),
 ('digital roadmap', 9)]